# 0. Set up and configuration

In [15]:
import numpy as np
import pandas as pd
from pathlib import Path
import yfinance as yf
from pykalman import KalmanFilter
import os
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

# 1. Introduction

This notebook presents our group’s final project and follows the same structure as our final report. Each section documents the methodology, analysis, results, and conclusions developed throughout the project.

# 2. Data Collection and Data Description

## 2.1 Data Source
   

In [ ]:
'''
Data preprocessing structure:
 1. download raw data
 2. pair each pair
 3. align trading dates
 4. drop missing observations within each pair
'''
BASE_DIR = "C:/Users/Serena/OneDrive - University of Illinois - Urbana/Desktop/Columbia/5291/Project"
DATA_DIR = BASE_DIR / "Data"
OUTPUT_DIR = DATA_DIR/"yfinance_data"
pair1_y = 'KO'
pair1_x = 'PEP'
pair2_y = 'XOM'
pair2_x = 'CVX'

start_date = '2010-01-01'
end_date = '2026-01-01'

data = yf.download([pair1_y, pair1_x, pair2_y, pair2_x],
                   start=start_date, end=end_date,
                   auto_adjust=False)['Adj Close']

KO =  yf.download(pair1_y, start=start_date, end=end_date, auto_adjust=False)['Adj Close']
PEP = yf.download(pair1_x, start=start_date, end=end_date, auto_adjust=False)['Adj Close']
XOM = yf.download(pair2_y, start=start_date, end=end_date, auto_adjust=False)['Adj Close']
CVX = yf.download(pair2_x, start=start_date, end=end_date, auto_adjust=False)['Adj Close']

KO.to_csv(OUTPUT_DIR/"KO_raw.csv")
PEP.to_csv(OUTPUT_DIR/"PEP_raw.csv")
XOM.to_csv(OUTPUT_DIR/"XOM_raw.csv")
CVX.to_csv(OUTPUT_DIR/"CVX_raw.csv")

# 2.2 Pair each pair and align dates, clean data

In [ ]:
#pair and align the dates and drop missing observations
KO_PEP = pd.concat([KO, PEP], axis=1)
KO_PEP.columns = ['KO', 'PEP']

XOM_CVX = pd.concat([XOM, CVX], axis=1)
XOM_CVX.columns = ['XOM', 'CVX']

KO_PEP = KO_PEP.dropna()
XOM_CVX = XOM_CVX.dropna()

KO_PEP.to_csv(OUTPUT_DIR/"KO_PEP.csv")
XOM_CVX.to_csv(OUTPUT_DIR/"XOM_CVX.csv")

## 2.3 Preprocessing - Compute log prices and log returns

In [ ]:
'''
log price and log return structure:
1. from pair data, get log price and log return
2. drop missing observations 
'''
BASE_DIR = BASE_DIR = "C:/Users/Serena/OneDrive - University of Illinois - Urbana/Desktop/Columbia/5291/Project"
DATA_DIR = BASE_DIR + "/Data"
INPUT_DIR = DATA_DIR + "/yfinance_data"
OUTPUT_DIR = DATA_DIR + "/log_price_log_return"

KO_PEP  = pd.read_csv(INPUT_DIR + "/KO_PEP.csv")
XOM_CVX = pd.read_csv(INPUT_DIR + "/XOM_CVX.csv")

KO_PEP = KO_PEP.set_index("Date")
XOM_CVX = XOM_CVX.set_index("Date")

KO_log_prices = np.log(KO_PEP['KO'])
PEP_log_prices = np.log(KO_PEP['PEP'])
XOM_log_prices = np.log(XOM_CVX['XOM'])
CVX_log_prices = np.log(XOM_CVX['CVX'])

KO_log_returns = KO_log_prices.diff().dropna()
PEP_log_returns = PEP_log_prices.diff().dropna()
XOM_log_returns = XOM_log_prices.diff().dropna()
CVX_log_returns = CVX_log_prices.diff().dropna()

KO_log_prices.to_csv(OUTPUT_DIR + "/KO_log_prices.csv")
PEP_log_prices.to_csv(OUTPUT_DIR + "/PEP_log_prices.csv")
XOM_log_prices.to_csv(OUTPUT_DIR + "/XOM_log_prices.csv")
CVX_log_prices.to_csv(OUTPUT_DIR + "/CVX_log_prices.csv")

KO_log_returns.to_csv(OUTPUT_DIR + "/KO_log_returns.csv")
PEP_log_returns.to_csv(OUTPUT_DIR + "/PEP_log_returns.csv")
XOM_log_returns.to_csv(OUTPUT_DIR + "/XOM_log_returns.csv")
CVX_log_returns.to_csv(OUTPUT_DIR + "/CVX_log_returns.csv")

## 2.4 Exploratory Data Analysis

## 2.4.1 Data Quality and Sample Coverage

In [ ]:
EDA_DIR = DATA_DIR + "/output/EDA output"
print("KO-PEP shape:", KO_PEP.shape)
print("XOM-CVX shape:", XOM_CVX.shape)

print(KO_PEP.head(), "KO-PEP raw pair data")
print(XOM_CVX.head(), "XOM-CVX raw pair data")

In [ ]:
def combined_data_quality_summary(data_dict):
    rows = []

    for pair_name, df in data_dict.items():
        rows.append({
            "pair": pair_name,
            "start_date": df.index.min(),
            "end_date": df.index.max(),
            "n_obs": len(df),
            "columns": ", ".join(df.columns),
            "missing_total": df.isna().sum().sum(),
            "missing_pct_total": df.isna().sum().sum() / (df.shape[0] * df.shape[1])
        })

    return pd.DataFrame(rows)

data_quality_table = combined_data_quality_summary({
    "KO-PEP": KO_PEP,
    "XOM-CVX": XOM_CVX
})
data_quality_table.to_csv(
    (EDA_DIR + "/data_quality_summary.csv"),
    index=False)

## 2.4.2 Return correlation

In [ ]:
returns_ko_pep = pd.concat(
    [KO_log_returns.rename("KO"), PEP_log_returns.rename("PEP")],
    axis=1
).dropna()

returns_xom_cvx = pd.concat(
    [XOM_log_returns.rename("XOM"), CVX_log_returns.rename("CVX")],
    axis=1
).dropna()

pair_corr = pd.DataFrame([
    {
        "pair": "KO-PEP",
        "return_correlation": returns_ko_pep["KO"].corr(returns_ko_pep["PEP"])
    },
    {
        "pair": "XOM-CVX",
        "return_correlation": returns_xom_cvx["XOM"].corr(returns_xom_cvx["CVX"])
    }
])

pair_corr

### 2.4.3 Price co-movement plots

In [25]:
all_log_prices = pd.concat(
    [
        KO_log_prices.rename("KO"),
        PEP_log_prices.rename("PEP"),
        XOM_log_prices.rename("XOM"),
        CVX_log_prices.rename("CVX")
    ],
    axis=1
).sort_index()
log_returns = pd.concat(
    [
        KO_log_returns.rename("KO"),
        PEP_log_returns.rename("PEP"),
        XOM_log_returns.rename("XOM"),
        CVX_log_returns.rename("CVX")
    ],
    axis=1
).sort_index()

In [ ]:
EDA_DIR = DATA_DIR + "/output/EDA output"

def save_and_show(fig, filename):
    filename = filename.lstrip("/\\")
    
    save_path = os.path.join(EDA_DIR, filename)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)


def plot_log_prices(log_price_df, cols, title, filename):
    fig, ax = plt.subplots(figsize=(10, 5))

    plot_df = log_price_df[cols].dropna()

    for col in cols:
        ax.plot(plot_df.index, plot_df[col], label=col)

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Log Adjusted Close")
    ax.legend()
    ax.grid(True, alpha=0.3)

    save_and_show(fig, filename)


def plot_normalized_prices(price_df, cols, title, filename):
    df = price_df[cols].dropna().copy()
    df.index = pd.to_datetime(df.index)

    normalized = df / df.iloc[0]

    fig, ax = plt.subplots(figsize=(10, 5))

    for col in cols:
        ax.plot(normalized.index, normalized[col], label=col)

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Normalized Price")
    ax.legend()
    ax.grid(True, alpha=0.3)

    save_and_show(fig, filename)

def plot_rolling_correlation(log_returns_df, col1, col2, window=252, filename=None):
    df = log_returns_df[[col1, col2]].dropna()
    rolling_corr = df[col1].rolling(window).corr(df[col2])

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(rolling_corr.index, rolling_corr, label=f"{window}-day rolling corr")
    ax.axhline(
        rolling_corr.mean(),
        linestyle="--",
        label=f"Mean = {rolling_corr.mean():.3f}"
    )

    ax.set_title(f"{col1}-{col2} Rolling Return Correlation ({window} days)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Rolling Correlation")
    ax.legend()
    ax.grid(True, alpha=0.3)

    if filename is not None:
        save_and_show(fig, filename)
    else:
        plt.show()

    return rolling_corr

In [ ]:
plot_log_prices(
    all_log_prices,
    ["KO", "PEP"],
    "KO-PEP Log Prices",
    "ko_pep_log_prices.png"
)

plot_log_prices(
    all_log_prices,
    ["XOM", "CVX"],
    "XOM-CVX Log Prices",
    "xom_cvx_log_prices.png"
)

plot_normalized_prices(
    KO_PEP,
    ["KO", "PEP"],
    "KO-PEP Normalized Prices",
    "ko_pep_normalized_prices.png"
)

plot_normalized_prices(
    XOM_CVX,
    ["XOM", "CVX"],
    "XOM-CVX Normalized Prices",
    "xom_cvx_normalized_prices.png"
)

rolling_corr_ko_pep = plot_rolling_correlation(
    log_returns,
    "KO",
    "PEP",
    window=252,
    filename="ko_pep_rolling_return_correlation.png"
)

rolling_corr_xom_cvx = plot_rolling_correlation(
    log_returns,
    "XOM",
    "CVX",
    window=252,
    filename="xom_cvx_rolling_return_correlation.png"
)

# 3. Statistical Models

## 3.1 Spread Construction Models

### 3.1.1 Static OLS spread and rolling z-score

In [ ]:
KO_log_prices = KO_log_prices.set_index("Date")
PEP_log_prices = PEP_log_prices.set_index("Date")
XOM_log_prices = XOM_log_prices.set_index("Date")
CVX_log_prices = CVX_log_prices.set_index("Date")   

In [ ]:
'''
ols spread and z-score structure:

static OLS regression to get static spread and rolling z-score
1 rolling windows: rolling z-score normalization window

y = alpha + beta * x
spread_t = y_t - (alpha_t + beta_t * x_t)

z-score_t = (spread_t - mean(spread_{t-252:t-1})) / std(spread_{t-252:t-1})
'''
def calculate_ols_spread_and_zscore(y, x, lookback=252):
    X = np.column_stack([np.ones(len(x)), x.values])
    Y = y.values

    coeffs = np.linalg.lstsq(X, Y, rcond=None)[0]
    alpha = coeffs[0]
    beta = coeffs[1]

    # static spread
    spread = y - (alpha + beta * x)

    # compute rolling z-score
    rolling_mean = spread.shift(1).rolling(window = lookback).mean()
    rolling_std = spread.shift(1).rolling(window = lookback).std()

    zscore = (spread - rolling_mean) / rolling_std

    return spread, zscore, beta, alpha

spread_ols_ko_pep, zscore_ols_ko_pep, beta_ols_ko_pep, alpha_ols_ko_pep = calculate_ols_spread_and_zscore(
    KO_log_prices['KO'], PEP_log_prices['PEP']
)

spread_ols_xom_cvx, zscore_ols_xom_cvx, beta_ols_xom_cvx, alpha_ols_xom_cvx = calculate_ols_spread_and_zscore(
    XOM_log_prices['XOM'], CVX_log_prices['CVX']
)

In [ ]:
''' 
input for backtest: Date, pair, return_y, return_x, OLS_spread, OLS_zscore, OLS_beta, OLS_alpha

will be used in backtest to generate trading signals
'''

def build_partial_backtest_input(pair_name, log_price_y, log_price_x, ols_spread, ols_zscore, ols_beta, ols_alpha):
    y_col, x_col = pair_name.split("-")

    df = pd.DataFrame(index=log_price_y.index)

    df["Date"] = df.index
    df["pair"] = pair_name

    df["return_y"] = log_price_y[y_col].diff()
    df["return_x"] = log_price_x[x_col].diff()

    df["OLS_spread"] = ols_spread
    df["OLS_zscore"] = ols_zscore
    df["OLS_beta"] = ols_beta
    df["OLS_alpha"] = ols_alpha

    return df.reset_index(drop=True)

ko_pep_backtest_input = build_partial_backtest_input(
    "KO-PEP", KO_log_prices, PEP_log_prices, spread_ols_ko_pep, zscore_ols_ko_pep, beta_ols_ko_pep, alpha_ols_ko_pep)
xom_cvx_backtest_input = build_partial_backtest_input(
    "XOM-CVX", XOM_log_prices, CVX_log_prices, spread_ols_xom_cvx, zscore_ols_xom_cvx, beta_ols_xom_cvx, alpha_ols_xom_cvx)

## 3.1.2 Kalman dynamic spread and z-scores

In [ ]:
''' 
kalman model: 

y_t = alpha_t + beta_t x_t + epsilon_t

dynamic spread:

spread_kalman = y - alpha_pred - beta_pred * x

rolling z-score:

z_kalman = (spread_kalman - spread_kalman.rolling(252).mean()) / spread_kalman.rolling(252).std()

Step1: use yesterday's alpha and beta to predict today's
alpha_t|t-1
beta_t|t-1

Step2: compute today's spread
spread_kalman = y - alpha_pred - beta_pred * x

Step3: observe today's y and update kalman state to get
alpha_t|t
beta_t|t
for tomorrow's use

1. predict beta_t
2. compute spread_t
3. compute zscore_t
4. generate signal_t
5. observe y_t
6. update beta_{t+1}
'''

def calculate_kalman_spread_and_zscore(y, x, lookback=252):
    alpha_kf = pd.Series(index=y.index, dtype=float)
    beta_kf = pd.Series(index=y.index, dtype=float)
    spread_kf = pd.Series(index=y.index, dtype=float)

    state_mean = np.array([0.0, 1.0])
    state_cov = np.eye(2)

    Q = 0.0001 * np.eye(2)
    R = 0.01

    for i in range(len(y)):
        obs_matrix = np.array([[1.0, x.iloc[i]]])
        state_mean_pred = state_mean
        state_cov_pred = state_cov + Q

        # use predicted alpha/beta to calculate current spread
        alpha_pred = state_mean_pred[0]
        beta_pred = state_mean_pred[1]

        alpha_kf.iloc[i] = alpha_pred
        beta_kf.iloc[i] = beta_pred
        spread_kf.iloc[i] = y.iloc[i] - (alpha_pred + beta_pred * x.iloc[i])

        # update step using current y
        y_obs = y.iloc[i]
        y_pred = obs_matrix @ state_mean_pred
        innovation = y_obs - y_pred
        S = obs_matrix @ state_cov_pred @ obs_matrix.T + R
        K = state_cov_pred @ obs_matrix.T / S

        state_mean = state_mean_pred + K.flatten() * innovation
        state_cov = state_cov_pred - K @ obs_matrix @ state_cov_pred

    rolling_mean = spread_kf.shift(1).rolling(lookback).mean()
    rolling_std = spread_kf.shift(1).rolling(lookback).std()

    zscore_kf = (spread_kf - rolling_mean) / rolling_std

    return spread_kf, zscore_kf, beta_kf, alpha_kf

spread_kf_ko_pep, zscore_kf_ko_pep, beta_kf_ko_pep, alpha_kf_ko_pep = calculate_kalman_spread_and_zscore(
    KO_log_prices['KO'], PEP_log_prices['PEP']
)

spread_kf_xom_cvx, zscore_kf_xom_cvx, beta_kf_xom_cvx, alpha_kf_xom_cvx = calculate_kalman_spread_and_zscore(
    XOM_log_prices['XOM'], CVX_log_prices['CVX']
)

In [ ]:
ko_pep_backtest_input = build_partial_backtest_input(
    "KO-PEP", KO_log_prices, PEP_log_prices, spread_ols_ko_pep, zscore_ols_ko_pep, beta_ols_ko_pep, alpha_ols_ko_pep)
xom_cvx_backtest_input = build_partial_backtest_input(
    "XOM-CVX", XOM_log_prices, CVX_log_prices, spread_ols_xom_cvx, zscore_ols_xom_cvx, beta_ols_xom_cvx, alpha_ols_xom_cvx)

In [ ]:
''' 
input for backtest, will be used in backtest to generate trading signals
'''
def add_kalman_columns(
    backtest_df,
    kf_spread,
    kf_zscore,
    kf_beta,
    kf_alpha
):
    df = backtest_df.copy()

    df["Date"] = pd.to_datetime(df["Date"])

    kf_df = pd.DataFrame({
        "Date": kf_spread.index,
        "Kalman_spread": kf_spread.values,
        "Kalman_zscore": kf_zscore.values,
        "Kalman_beta_t": kf_beta.values,
        "Kalman_alpha_t": kf_alpha.values
    })

    kf_df["Date"] = pd.to_datetime(kf_df["Date"])

    df = df.merge(kf_df, on="Date", how="left")

    return df

In [ ]:
ko_pep_backtest_input = add_kalman_columns(
    ko_pep_backtest_input,
    spread_kf_ko_pep,
    zscore_kf_ko_pep,
    beta_kf_ko_pep,
    alpha_kf_ko_pep
)

In [ ]:
xom_cvx_backtest_input = add_kalman_columns(
    xom_cvx_backtest_input,
    spread_kf_xom_cvx,
    zscore_kf_xom_cvx,
    beta_kf_xom_cvx,
    alpha_kf_xom_cvx
)

In [ ]:
backtest_input = pd.concat(
    [ko_pep_backtest_input, xom_cvx_backtest_input],
    axis=0
).sort_values(["pair", "Date"])

backtest_input.to_csv(OUTPUT_DIR / "backtest_input_with_kalman.csv", index=False)

## 3.2 ML Filter Model Design

### 3.2.1 PnL-Based Entry Label

In [ ]:
"""
    PnL-based ML label.

    z > 2  : short spread = short Y, long beta * X
    z < -2 : long spread  = long Y, short beta * X

    Label = 1 if 10-day forward trade PnL proxy exceeds min_return.
"""
def align_to_z_index(x, z_index, name=None):
    if np.isscalar(x):
        return pd.Series(x, index=z_index, dtype=float, name=name)

    s = pd.Series(x).astype(float)

    # Case 1: already same index
    if s.index.equals(z_index):
        return s.rename(name)

    # Case 2: same length, force same index
    if len(s) == len(z_index):
        s = s.copy()
        s.index = z_index
        return s.rename(name)

    # Case 3: try normal reindex
    s_reindexed = s.reindex(z_index)

    # Case 4: if reindex failed badly, align overlapping tail by position
    if s_reindexed.notna().sum() == 0:
        out = pd.Series(index=z_index, dtype=float, name=name)
        n = min(len(s), len(z_index))
        out.iloc[-n:] = s.iloc[-n:].values
        return out

    return s_reindexed.rename(name)

def create_pnl_trade_labels(
    zscore,
    log_returns_y,
    log_returns_x,
    beta,
    horizon=10,
    entry_threshold=2.0,
    min_return=0.0
):

    z = pd.Series(zscore).astype(float)
    idx = z.index

    ry = align_to_z_index(log_returns_y, idx, "ret_y")
    rx = align_to_z_index(log_returns_x, idx, "ret_x")
    beta_series = align_to_z_index(beta, idx, "beta").ffill()

    labels = pd.Series(index=idx, dtype=float)
    pnl_proxy = pd.Series(index=idx, dtype=float)

    for i in range(len(z) - horizon):
        cur_z = z.iloc[i]

        if pd.isna(cur_z):
            continue

        if abs(cur_z) < entry_threshold:
            continue

        b = beta_series.iloc[i]

        if pd.isna(b):
            continue

        future_ry = ry.iloc[i + 1 : i + horizon + 1]
        future_rx = rx.iloc[i + 1 : i + horizon + 1]

        if future_ry.notna().sum() < horizon * 0.8:
            continue
        if future_rx.notna().sum() < horizon * 0.8:
            continue

        # Spread return: long Y, short beta X
        spread_return = future_ry.sum() - b * future_rx.sum()

        # If z > 2, we short the spread.
        # If z < -2, we long the spread.
        direction = -np.sign(cur_z)

        trade_pnl = direction * spread_return

        pnl_proxy.iloc[i] = trade_pnl
        labels.iloc[i] = 1.0 if trade_pnl > min_return else 0.0

    return labels, pnl_proxy

In [ ]:
labels_ols_ko_pep, pnl_ols_ko_pep = create_pnl_trade_labels(
    zscore_ols_ko_pep,
    KO_log_returns["KO"],
    PEP_log_returns["PEP"],
    beta_ols_ko_pep,
    horizon=10,
    entry_threshold=2.0,
    min_return=0.0
)

labels_kf_ko_pep, pnl_kf_ko_pep = create_pnl_trade_labels(
    zscore_kf_ko_pep,
    KO_log_returns["KO"],
    PEP_log_returns["PEP"],
    beta_kf_ko_pep,
    horizon=10,
    entry_threshold=2.0,
    min_return=0.0
)

labels_ols_xom_cvx, pnl_ols_xom_cvx = create_pnl_trade_labels(
    zscore_ols_xom_cvx,
    XOM_log_returns["XOM"],
    CVX_log_returns["CVX"],
    beta_ols_xom_cvx,
    horizon=10,
    entry_threshold=2.0,
    min_return=0.0
)

labels_kf_xom_cvx, pnl_kf_xom_cvx = create_pnl_trade_labels(
    zscore_kf_xom_cvx,
    XOM_log_returns["XOM"],
    CVX_log_returns["CVX"],
    beta_kf_xom_cvx,
    horizon=10,
    entry_threshold=2.0,
    min_return=0.0
)

In [ ]:
for name, lab in {
    "OLS KO-PEP": labels_ols_ko_pep,
    "KF KO-PEP": labels_kf_ko_pep,
    "OLS XOM-CVX": labels_ols_xom_cvx,
    "KF XOM-CVX": labels_kf_xom_cvx,
}.items():
    print(name)
    print(lab.value_counts(dropna=True))
    print("positive ratio:", lab.dropna().mean())
    print("labeled samples:", lab.notna().sum())
    print()

### 3.2.2 Feature Construction

In [ ]:
#features 
def calculate_rolling_features(
    spread,
    zscore,
    log_returns_y,
    log_returns_x,
    beta=None,
    window=20,
    adf_window=60
):
    from statsmodels.tsa.stattools import adfuller
    spread = pd.Series(spread).astype(float)
    zscore = pd.Series(zscore).reindex(spread.index).astype(float)

    # Align returns with spread index
    log_returns_y = pd.Series(log_returns_y).reindex(spread.index).astype(float)
    log_returns_x = pd.Series(log_returns_x).reindex(spread.index).astype(float)

    features = pd.DataFrame(index=spread.index)

    # Current signal state
    features["spread"] = spread
    features["zscore"] = zscore
    features["abs_zscore"] = zscore.abs()

    # Beta features
    if beta is not None:
        if np.isscalar(beta):
            beta = pd.Series(beta, index=spread.index, dtype=float)
        else:
            beta = pd.Series(beta).reindex(spread.index).ffill().astype(float)

        features["beta"] = beta
        features["delta_beta"] = beta.diff()
        features["abs_delta_beta"] = features["delta_beta"].abs()

    # Z-score lags and momentum
    features["zscore_lag1"] = zscore.shift(1)
    features["zscore_lag3"] = zscore.shift(3)
    features["zscore_lag5"] = zscore.shift(5)
    features["delta_zscore"] = zscore.diff()

    # Use previous window to avoid same-day rolling-stat leakage
    spread_hist = spread.shift(1)
    ret_y_hist = log_returns_y.shift(1)
    ret_x_hist = log_returns_x.shift(1)

    features["spread_volatility"] = spread_hist.rolling(
        window, min_periods=int(window * 0.8)
    ).std()

    features["spread_autocorr"] = spread_hist.rolling(
        window, min_periods=int(window * 0.8)
    ).apply(lambda x: pd.Series(x).autocorr(lag=1), raw=False)

    features["return_correlation"] = ret_y_hist.rolling(
        window, min_periods=int(window * 0.8)
    ).corr(ret_x_hist)

    def rolling_adf_pvalue(series, window=60):
        pvals = pd.Series(index=series.index, dtype=float)
        s = series.astype(float)

        for i in range(window, len(s)):
            w = s.iloc[i-window:i].dropna()

            if len(w) < window * 0.9:
                continue
            if w.nunique() < 3:
                continue

            try:
                pvals.iloc[i] = adfuller(
                    w.values,
                    maxlag=1,
                    regression="c",
                    autolag=None
                )[1]
            except Exception:
                pvals.iloc[i] = np.nan

        return pvals
    features["adf_pvalue"] = rolling_adf_pvalue(spread, window=adf_window)

    return features

### 3.2.3 Feature Sets

In [ ]:
# ---------------------------------------------------
# Feature sets for model selection
# ---------------------------------------------------

full_features = [
    "spread",
    "zscore",
    "abs_zscore",
    "zscore_lag1",
    "zscore_lag3",
    "zscore_lag5",
    "delta_zscore",
    "spread_volatility",
    "spread_autocorr",
    "adf_pvalue",
    "beta",
    "delta_beta",
    "abs_delta_beta"
]

reduced_features = [
    "abs_zscore",
    "zscore_lag1",
    "zscore_lag3",
    "delta_zscore",
    "spread_volatility",
    "spread_autocorr",
    "adf_pvalue",
    "delta_beta",
    "abs_delta_beta"
]

minimal_features = [
    "abs_zscore",
    "delta_zscore",
    "spread_volatility",
    "adf_pvalue",
    "abs_delta_beta"
]

feature_sets = {
    "full": full_features,
    "reduced": reduced_features,
    "minimal": minimal_features,
}

## 3.3 Final ML Filter Selection

### 3.3.1 Candidate Models

In [ ]:
def build_model(model_type, params):
    if model_type == "tree":
        clf = DecisionTreeClassifier(
            criterion=params.get("criterion", "gini"),
            max_depth=params.get("max_depth", 2),
            min_samples_leaf=params.get("min_samples_leaf", 20),
            min_samples_split=params.get("min_samples_split", 20),
            max_features=params.get("max_features", None),
            ccp_alpha=params.get("ccp_alpha", 0.0),
            class_weight="balanced",
            random_state=42
        )
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", clf)
        ])

    elif model_type == "knn":
        clf = KNeighborsClassifier(
            n_neighbors=params.get("n_neighbors", 15),
            weights=params.get("weights", "distance"),
            metric=params.get("metric", "minkowski"),
            p=params.get("p", 2),
            leaf_size=params.get("leaf_size", 30)
        )

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", clf)
        ])

    elif model_type == "random_forest":
        clf = RandomForestClassifier(
            n_estimators=params.get("n_estimators", 100),
            criterion=params.get("criterion", "gini"),
            max_depth=params.get("max_depth", 3),
            min_samples_leaf=params.get("min_samples_leaf", 20),
            min_samples_split=params.get("min_samples_split", 20),
            max_features=params.get("max_features", "sqrt"),
            bootstrap=params.get("bootstrap", True),
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", clf)
        ])

    else:
        raise ValueError("model_type must be 'tree', 'knn', or 'random_forest'")

    return model

### 3.3.2 - 3.3.3 Extended Parameter Tuning and Final Selected Filters

In [ ]:
def clean_selected_features(train_df, feature_list, missing_threshold=0.5):
    selected = []

    for col in feature_list:
        if col not in train_df.columns:
            continue
        if train_df[col].isna().mean() > missing_threshold:
            continue
        if train_df[col].nunique(dropna=True) <= 1:
            continue

        selected.append(col)

    return selected

In [ ]:
model_grid = []

# Tree
for criterion in ["gini", "entropy"]:
    for depth in [1, 2, 3]:
        for leaf in [10, 20, 30, 40]:
            for ccp in [0.0, 0.001, 0.005]:
                model_grid.append({
                    "model_type": "tree",
                    "params": {
                        "criterion": criterion,
                        "max_depth": depth,
                        "min_samples_leaf": leaf,
                        "min_samples_split": 20,
                        "ccp_alpha": ccp,
                        "max_features": None
                    }
                })

# KNN
for k in [5, 10, 15, 20, 30]:
    for weights in ["uniform", "distance"]:
        for p in [1, 2]:
            model_grid.append({
                "model_type": "knn",
                "params": {
                    "n_neighbors": k,
                    "weights": weights,
                    "metric": "minkowski",
                    "p": p,
                    "leaf_size": 30
                }
            })

# Random Forest
for n_est in [100, 200]:
    for depth in [2, 3, 4]:
        for leaf in [10, 20, 30]:
            for max_feat in ["sqrt", "log2"]:
                model_grid.append({
                    "model_type": "random_forest",
                    "params": {
                        "n_estimators": n_est,
                        "criterion": "gini",
                        "max_depth": depth,
                        "min_samples_leaf": leaf,
                        "min_samples_split": 20,
                        "max_features": max_feat,
                        "bootstrap": True
                    }
                })

print("Total candidate models:", len(model_grid))

In [ ]:
def model_selection_for_dataset(
    train_df,
    val_df,
    test_df,
    dataset_name,
    model_grid,
    feature_sets,
    min_approval=0.20,
    max_approval=0.80
):
    y_train = train_df["label"].astype(int).values
    y_val = val_df["label"].astype(int).values
    y_test = test_df["label"].astype(int).values

    rows = []
    fitted_models = {}
    model_id = 0

    for feature_set_name, raw_feature_cols in feature_sets.items():

        feature_cols = clean_selected_features(
            train_df,
            raw_feature_cols,
            missing_threshold=0.5
        )
        if len(feature_cols) == 0:
            continue
        X_train = train_df[feature_cols]
        X_val = val_df[feature_cols]
        X_test = test_df[feature_cols]
        for spec in model_grid:
            model_type = spec["model_type"]
            params = spec["params"]

            model = build_model(model_type, params)
            model.fit(X_train, y_train)

            train_prob = model.predict_proba(X_train)[:, 1]
            val_prob = model.predict_proba(X_val)[:, 1]

            train_auc = roc_auc_score(y_train, train_prob) if len(np.unique(y_train)) > 1 else np.nan
            val_auc = roc_auc_score(y_val, val_prob) if len(np.unique(y_val)) > 1 else np.nan

            best_th_row, val_sweep = choose_best_threshold(
                y_val,
                val_prob,
                min_approval=min_approval,
                max_approval=max_approval,
                metric="balanced_accuracy"
            )
            auc_gap = train_auc - val_auc
            row = {
                "model_id": model_id,
                "dataset": dataset_name,
                "feature_set": feature_set_name,
                "n_features": len(feature_cols),
                "feature_cols": feature_cols,
                "model_type": model_type,
                "params": params,
                "train_auc": train_auc,
                "val_auc": val_auc,
                "auc_gap": auc_gap,
                "selected_threshold": best_th_row["threshold"],
                "val_approval_rate": best_th_row["approval_rate"],
                "val_balanced_accuracy": best_th_row["balanced_accuracy"],
                "val_precision": best_th_row["precision"],
                "val_recall": best_th_row["recall"],
                "val_f1": best_th_row["f1"],
            }
            rows.append(row)
            fitted_models[model_id] = {
                "model": model,
                "feature_cols": feature_cols,
                "feature_set": feature_set_name,
                "train_prob": train_prob,
                "val_prob": val_prob,
                "val_sweep": val_sweep,
            }
            model_id += 1

    results = pd.DataFrame(rows)

    if len(results) == 0:
        raise ValueError(f"No valid models were fitted for {dataset_name}.")

    # Stronger overfitting penalty
    results["selection_score"] = (
        0.50 * results["val_auc"].fillna(0)
        + 0.50 * results["val_balanced_accuracy"].fillna(0)
        - 0.50 * results["auc_gap"].clip(lower=0).fillna(0)
    )

    # Remove severe overfitting if possible
    results_filtered = results[results["auc_gap"] <= 0.25].copy()

    if len(results_filtered) == 0:
        results_filtered = results.copy()

    results_sorted = results_filtered.sort_values(
        ["selection_score", "val_auc", "val_balanced_accuracy", "val_f1"],
        ascending=[False, False, False, False]
    )

    best_row = results_sorted.iloc[0]
    best_id = int(best_row["model_id"])
    best_info = fitted_models[best_id]

    best_model = best_info["model"]
    best_feature_cols = best_info["feature_cols"]
    threshold = float(best_row["selected_threshold"])

    X_test_best = test_df[best_feature_cols]
    test_prob = best_model.predict_proba(X_test_best)[:, 1]
    test_metrics = compute_metrics(y_test, test_prob, threshold=threshold)

    y_test_pred = (test_prob >= threshold).astype(int)

    print("\n==============================")
    print(f"Best model for {dataset_name}")
    print(f"Feature set: {best_row['feature_set']}")
    print(f"Number of features: {best_row['n_features']}")
    print(f"Features: {best_feature_cols}")
    print(f"Model type: {best_row['model_type']}")
    print(f"Params: {best_row['params']}")
    print(f"Selected threshold: {threshold:.2f}")
    print(f"Train AUC: {best_row['train_auc']:.3f}")
    print(f"Val AUC: {best_row['val_auc']:.3f}")
    print(f"AUC gap: {best_row['auc_gap']:.3f}")

    print("\nValidation metrics:")
    print(best_row[[
        "val_approval_rate",
        "val_balanced_accuracy",
        "val_precision",
        "val_recall",
        "val_f1"
    ]])

    print("\nTest metrics:")
    print(pd.Series(test_metrics))

    print("\nTest confusion matrix:")
    print(confusion_matrix(y_test, y_test_pred))

    return {
        "dataset_name": dataset_name,
        "results": results_sorted,
        "all_results": results,
        "best_row": best_row,
        "best_model": best_model,
        "feature_cols": best_feature_cols,
        "feature_set": best_row["feature_set"],
        "threshold": threshold,
        "test_prob": test_prob,
        "test_metrics": test_metrics,
        "test_pred": y_test_pred,
    }

In [ ]:
selection_ols_ko = model_selection_for_dataset(
    train_ols_ko,
    val_ols_ko,
    test_ols_ko,
    dataset_name="OLS KO-PEP",
    model_grid=model_grid,
    feature_sets=feature_sets,
    min_approval=0.20,
    max_approval=0.80
)

selection_kf_ko = model_selection_for_dataset(
    train_kf_ko,
    val_kf_ko,
    test_kf_ko,
    dataset_name="KF KO-PEP",
    model_grid=model_grid,
    feature_sets=feature_sets,
    min_approval=0.20,
    max_approval=0.80
)

selection_ols_xom = model_selection_for_dataset(
    train_ols_xom,
    val_ols_xom,
    test_ols_xom,
    dataset_name="OLS XOM-CVX",
    model_grid=model_grid,
    feature_sets=feature_sets,
    min_approval=0.20,
    max_approval=0.80
)

selection_kf_xom = model_selection_for_dataset(
    train_kf_xom,
    val_kf_xom,
    test_kf_xom,
    dataset_name="KF XOM-CVX",
    model_grid=model_grid,
    feature_sets=feature_sets,
    min_approval=0.20,
    max_approval=0.80
)

In [ ]:
#output summary of selected models for backtest
selected_model_summary = pd.DataFrame([
    {
        "dataset": selection_ols_ko["dataset_name"],
        "pair": "KO-PEP",
        "spread_model": "OLS",
        "selected_model": selection_ols_ko["best_row"]["model_type"],
        "feature_set": selection_ols_ko["feature_set"],
        "n_features": len(selection_ols_ko["feature_cols"]),
        "features": ", ".join(selection_ols_ko["feature_cols"]),
        "params": str(selection_ols_ko["best_row"]["params"]),
        "threshold": selection_ols_ko["threshold"],
        "test_auc": selection_ols_ko["test_metrics"]["auc"],
        "test_balanced_accuracy": selection_ols_ko["test_metrics"]["balanced_accuracy"],
        "test_precision": selection_ols_ko["test_metrics"]["precision"],
        "test_recall": selection_ols_ko["test_metrics"]["recall"],
        "test_f1": selection_ols_ko["test_metrics"]["f1"],
        "test_approval_rate": selection_ols_ko["test_metrics"]["approval_rate"],
    },
    {
        "dataset": selection_kf_ko["dataset_name"],
        "pair": "KO-PEP",
        "spread_model": "Kalman",
        "selected_model": selection_kf_ko["best_row"]["model_type"],
        "feature_set": selection_kf_ko["feature_set"],
        "n_features": len(selection_kf_ko["feature_cols"]),
        "features": ", ".join(selection_kf_ko["feature_cols"]),
        "params": str(selection_kf_ko["best_row"]["params"]),
        "threshold": selection_kf_ko["threshold"],
        "test_auc": selection_kf_ko["test_metrics"]["auc"],
        "test_balanced_accuracy": selection_kf_ko["test_metrics"]["balanced_accuracy"],
        "test_precision": selection_kf_ko["test_metrics"]["precision"],
        "test_recall": selection_kf_ko["test_metrics"]["recall"],
        "test_f1": selection_kf_ko["test_metrics"]["f1"],
        "test_approval_rate": selection_kf_ko["test_metrics"]["approval_rate"],
    },
    {
        "dataset": selection_ols_xom["dataset_name"],
        "pair": "XOM-CVX",
        "spread_model": "OLS",
        "selected_model": selection_ols_xom["best_row"]["model_type"],
        "feature_set": selection_ols_xom["feature_set"],
        "n_features": len(selection_ols_xom["feature_cols"]),
        "features": ", ".join(selection_ols_xom["feature_cols"]),
        "params": str(selection_ols_xom["best_row"]["params"]),
        "threshold": selection_ols_xom["threshold"],
        "test_auc": selection_ols_xom["test_metrics"]["auc"],
        "test_balanced_accuracy": selection_ols_xom["test_metrics"]["balanced_accuracy"],
        "test_precision": selection_ols_xom["test_metrics"]["precision"],
        "test_recall": selection_ols_xom["test_metrics"]["recall"],
        "test_f1": selection_ols_xom["test_metrics"]["f1"],
        "test_approval_rate": selection_ols_xom["test_metrics"]["approval_rate"],
    },
    {
        "dataset": selection_kf_xom["dataset_name"],
        "pair": "XOM-CVX",
        "spread_model": "Kalman",
        "selected_model": selection_kf_xom["best_row"]["model_type"],
        "feature_set": selection_kf_xom["feature_set"],
        "n_features": len(selection_kf_xom["feature_cols"]),
        "features": ", ".join(selection_kf_xom["feature_cols"]),
        "params": str(selection_kf_xom["best_row"]["params"]),
        "threshold": selection_kf_xom["threshold"],
        "test_auc": selection_kf_xom["test_metrics"]["auc"],
        "test_balanced_accuracy": selection_kf_xom["test_metrics"]["balanced_accuracy"],
        "test_precision": selection_kf_xom["test_metrics"]["precision"],
        "test_recall": selection_kf_xom["test_metrics"]["recall"],
        "test_f1": selection_kf_xom["test_metrics"]["f1"],
        "test_approval_rate": selection_kf_xom["test_metrics"]["approval_rate"],
    },
])

selected_model_summary.to_csv("selected_model_summary.csv", index=False)

selected_model_summary

In [ ]:
#output trading strategy signals for backtest
def create_selected_model_signal_file(dataset, selection_result, pair, spread_model):
    model = selection_result["best_model"]
    feature_cols = selection_result["feature_cols"]
    threshold = selection_result["threshold"]
    model_type = selection_result["best_row"]["model_type"]
    feature_set = selection_result["feature_set"]

    X = dataset[feature_cols]
    prob = model.predict_proba(X)[:, 1]
    signal = (prob >= threshold).astype(int)

    output = pd.DataFrame(index=dataset.index)

    output["date"] = output.index
    output["pair"] = pair
    output["spread_model"] = spread_model
    output["ml_model"] = model_type
    output["feature_set"] = feature_set
    output["n_features"] = len(feature_cols)

    output["z_score"] = dataset["zscore"]
    output["label"] = dataset["label"]

    output["prediction_probability"] = prob
    output["threshold"] = threshold
    output["ML_signal"] = signal

    return output.reset_index(drop=True)

In [ ]:
signals_ols_ko = create_selected_model_signal_file(
    dataset_ols_ko_pep,
    selection_ols_ko,
    pair="KO-PEP",
    spread_model="OLS"
)

signals_kf_ko = create_selected_model_signal_file(
    dataset_kf_ko_pep,
    selection_kf_ko,
    pair="KO-PEP",
    spread_model="Kalman"
)

signals_ols_xom = create_selected_model_signal_file(
    dataset_ols_xom_cvx,
    selection_ols_xom,
    pair="XOM-CVX",
    spread_model="OLS"
)

signals_kf_xom = create_selected_model_signal_file(
    dataset_kf_xom_cvx,
    selection_kf_xom,
    pair="XOM-CVX",
    spread_model="Kalman"
)

In [ ]:
OUTPUT_DIR = DATA_DIR + "/output"
ml_signals_final = pd.concat(
    [
        signals_ols_ko,
        signals_kf_ko,
        signals_ols_xom,
        signals_kf_xom,
    ],
    axis=0
).sort_values(["pair", "spread_model", "date"])

ml_signals_final.to_csv(OUTPUT_DIR + "/ml_signals_shallow.csv", index=False)

ml_signals_final.head()

In [ ]:
for name, df in {
    "OLS KO-PEP": signals_ols_ko,
    "KF KO-PEP": signals_kf_ko,
    "OLS XOM-CVX": signals_ols_xom,
    "KF XOM-CVX": signals_kf_xom,
}.items():
    print(name)
    print("rows:", len(df))
    print("model:", df["ml_model"].iloc[0])
    print("feature set:", df["feature_set"].iloc[0])
    print("threshold:", df["threshold"].iloc[0])
    print("approval rate:", df["ML_signal"].mean())
    print("avg prediction probability:", df["prediction_probability"].mean())
    print("label positive ratio:", df["label"].mean())
    print()

## 3.4 Trading Strategy and Backtest Model

In [ ]:
BACKTEST_INPUT_PATH = r"C:/Users/Serena/OneDrive - University of Illinois - Urbana/Desktop/Columbia/5291/Project/Data/output/backtest_input_with_kalman.csv"

backtest_input = pd.read_csv(BACKTEST_INPUT_PATH)
backtest_input["Date"] = pd.to_datetime(backtest_input["Date"])

print(backtest_input.shape)
print(backtest_input.columns.tolist())
backtest_input.head()

In [ ]:
def make_long_backtest_input(backtest_input):
    df = backtest_input.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    ols = df[[
        "Date",
        "pair",
        "return_y",
        "return_x",
        "OLS_spread",
        "OLS_zscore",
        "OLS_beta",
        "OLS_alpha"
    ]].copy()

    ols = ols.rename(columns={
        "Date": "date",
        "return_y": "ret_y",
        "return_x": "ret_x",
        "OLS_spread": "spread",
        "OLS_zscore": "zscore",
        "OLS_beta": "beta",
        "OLS_alpha": "alpha"
    })

    ols["spread_model"] = "OLS"

    kf = df[[
        "Date",
        "pair",
        "return_y",
        "return_x",
        "Kalman_spread",
        "Kalman_zscore",
        "Kalman_beta_t",
        "Kalman_alpha_t"
    ]].copy()

    kf = kf.rename(columns={
        "Date": "date",
        "return_y": "ret_y",
        "return_x": "ret_x",
        "Kalman_spread": "spread",
        "Kalman_zscore": "zscore",
        "Kalman_beta_t": "beta",
        "Kalman_alpha_t": "alpha"
    })

    kf["spread_model"] = "Kalman"

    out = pd.concat([ols, kf], axis=0)
    out = out.sort_values(["pair", "spread_model", "date"])
    out = out.dropna(subset=["zscore", "beta", "ret_y", "ret_x"])

    return out

backtest_long = make_long_backtest_input(backtest_input)

print(backtest_long.shape)
print(backtest_long.groupby(["pair", "spread_model"]).size())
backtest_long.head()

In [ ]:
def get_strategy_input(df, pair, spread_model):
    out = df[
        (df["pair"] == pair) &
        (df["spread_model"] == spread_model)
    ].copy()

    out = out.sort_values("date").set_index("date")
    return out

df_ols_ko = get_strategy_input(backtest_long, "KO-PEP", "OLS")
df_kf_ko = get_strategy_input(backtest_long, "KO-PEP", "Kalman")

df_ols_xom = get_strategy_input(backtest_long, "XOM-CVX", "OLS")
df_kf_xom = get_strategy_input(backtest_long, "XOM-CVX", "Kalman")

for name, df in {
    "OLS KO-PEP": df_ols_ko,
    "KF KO-PEP": df_kf_ko,
    "OLS XOM-CVX": df_ols_xom,
    "KF XOM-CVX": df_kf_xom,
}.items():
    print(name)
    print(df.shape)
    print(df[["spread", "zscore", "beta", "ret_y", "ret_x"]].isna().sum())
    print()

In [ ]:
ML_SIGNAL_PATH = r"C:/Users/Serena/OneDrive - University of Illinois - Urbana/Desktop/Columbia/5291/Project/Data/output/ml_signals_shallow.csv"

ml_signals = pd.read_csv(ML_SIGNAL_PATH)
ml_signals["date"] = pd.to_datetime(ml_signals["date"])

In [ ]:
def add_ml_signal(df, ml_signals, pair, spread_model):
    out = df.copy().reset_index()
    out["date"] = pd.to_datetime(out["date"])

    sig = ml_signals[
        (ml_signals["pair"] == pair) &
        (ml_signals["spread_model"] == spread_model)
    ][[
        "date",
        "ML_signal",
        "prediction_probability",
        "threshold",
        "ml_model",
        "feature_set"
    ]].copy()

    sig["date"] = pd.to_datetime(sig["date"])

    out = out.merge(sig, on="date", how="left")

    # Missing ML_signal means ML does not approve trade
    out["ML_signal"] = out["ML_signal"].fillna(0).astype(int)

    return out.sort_values("date").set_index("date")

df_ols_ko_ml = add_ml_signal(df_ols_ko, ml_signals, "KO-PEP", "OLS")
df_kf_ko_ml = add_ml_signal(df_kf_ko, ml_signals, "KO-PEP", "Kalman")

df_ols_xom_ml = add_ml_signal(df_ols_xom, ml_signals, "XOM-CVX", "OLS")
df_kf_xom_ml = add_ml_signal(df_kf_xom, ml_signals, "XOM-CVX", "Kalman")

In [ ]:
df_ols_ko_ml.head()

In [ ]:
for name, df in {
    "OLS KO-PEP ML": df_ols_ko_ml,
    "KF KO-PEP ML": df_kf_ko_ml,
    "OLS XOM-CVX ML": df_ols_xom_ml,
    "KF XOM-CVX ML": df_kf_xom_ml,
}.items():
    print(name)
    print("shape:", df.shape)
    print("ML approval rate:", df["ML_signal"].mean())
    print("ML signal count:", df["ML_signal"].sum())
    print()

In [ ]:
"""
    Backtest one pair strategy structure:
    Required columns:
        zscore
        beta
        ret_y
        ret_x
        ML_signal, only used when use_ml=True

    Position meaning:
        +1 = long spread  = long Y, short beta * X
        -1 = short spread = short Y, long beta * X
         0 = flat
    """
def backtest_pair_strategy(
    df,
    pair,
    strategy_name,
    use_ml=False,
    entry_threshold=2.0,
    exit_threshold=0.5,
    max_holding=20,
    cost_bps=5
):
    df = df.copy().sort_index()
    cost_rate = cost_bps / 10000
    position = 0
    holding_days = 0

    positions = []
    weight_y_list = []
    weight_x_list = []
    trade_change_list = []
    entry_signal_list = []
    exit_signal_list = []
    holding_days_list = []

    for date, row in df.iterrows():
        z = row["zscore"]
        beta = row["beta"]

        old_position = position
        entry_signal = 0
        exit_signal = 0
        # Exit rule
        if position != 0:
            holding_days += 1

            if abs(z) < exit_threshold:
                position = 0
                holding_days = 0
                exit_signal = 1

            elif holding_days > max_holding:
                position = 0
                holding_days = 0
                exit_signal = 1
        # Entry rule
        if position == 0:
            allow_trade = True

            if use_ml:
                allow_trade = row.get("ML_signal", 0) == 1

            if allow_trade:
                if z < -entry_threshold:
                    position = 1       # long spread
                    holding_days = 1
                    entry_signal = 1

                elif z > entry_threshold:
                    position = -1      # short spread
                    holding_days = 1
                    entry_signal = -1
        # Convert spread position to asset weights
        if position == 0 or pd.isna(beta):
            weight_y = 0.0
            weight_x = 0.0
        else:
            gross = 1 + abs(beta)
            weight_y = position / gross
            weight_x = -position * beta / gross

        trade_change = abs(position - old_position)

        positions.append(position)
        weight_y_list.append(weight_y)
        weight_x_list.append(weight_x)
        trade_change_list.append(trade_change)
        entry_signal_list.append(entry_signal)
        exit_signal_list.append(exit_signal)
        holding_days_list.append(holding_days)

    df["pair"] = pair
    df["strategy"] = strategy_name
    df["position"] = positions
    df["weight_y"] = weight_y_list
    df["weight_x"] = weight_x_list
    df["trade_change"] = trade_change_list
    df["entry_signal"] = entry_signal_list
    df["exit_signal"] = exit_signal_list
    df["holding_days"] = holding_days_list

    # avoid look-ahead bias
    df["weight_y_lag"] = df["weight_y"].shift(1).fillna(0)
    df["weight_x_lag"] = df["weight_x"].shift(1).fillna(0)

    # Strategy return
    df["gross_return"] = (
        df["weight_y_lag"] * df["ret_y"] +
        df["weight_x_lag"] * df["ret_x"]
    )

    # Turnover-based transaction cost
    df["turnover"] = (
        df["weight_y"].diff().abs().fillna(df["weight_y"].abs()) +
        df["weight_x"].diff().abs().fillna(df["weight_x"].abs())
    )

    df["transaction_cost"] = cost_rate * df["turnover"]
    df["daily_return"] = df["gross_return"] - df["transaction_cost"]

    df["cumulative_return"] = (1 + df["daily_return"]).cumprod() - 1

    return df

In [ ]:
#KO_PEP
results_s1_ko = backtest_pair_strategy(
    df_ols_ko,
    pair="KO-PEP",
    strategy_name="S1_OLS_zscore",
    use_ml=False,
    cost_bps=5
)

results_s2_ko = backtest_pair_strategy(
    df_kf_ko,
    pair="KO-PEP",
    strategy_name="S2_Kalman_zscore",
    use_ml=False,
    cost_bps=5
)

results_s3_ko = backtest_pair_strategy(
    df_ols_ko_ml,
    pair="KO-PEP",
    strategy_name="S3_OLS_ML",
    use_ml=True,
    cost_bps=5
)

results_s4_ko = backtest_pair_strategy(
    df_kf_ko_ml,
    pair="KO-PEP",
    strategy_name="S4_Kalman_ML",
    use_ml=True,
    cost_bps=5
)

In [ ]:
#XOM_CVX
results_s1_xom = backtest_pair_strategy(
    df_ols_xom,
    pair="XOM-CVX",
    strategy_name="S1_OLS_zscore",
    use_ml=False,
    cost_bps=5
)

results_s2_xom = backtest_pair_strategy(
    df_kf_xom,
    pair="XOM-CVX",
    strategy_name="S2_Kalman_zscore",
    use_ml=False,
    cost_bps=5
)

results_s3_xom = backtest_pair_strategy(
    df_ols_xom_ml,
    pair="XOM-CVX",
    strategy_name="S3_OLS_ML",
    use_ml=True,
    cost_bps=5
)

results_s4_xom = backtest_pair_strategy(
    df_kf_xom_ml,
    pair="XOM-CVX",
    strategy_name="S4_Kalman_ML",
    use_ml=True,
    cost_bps=5
)

In [ ]:
#Performance metrics
def compute_performance_metrics(df):
    daily_ret = df["daily_return"].dropna()

    if len(daily_ret) == 0:
        return {}

    total_return = (1 + daily_ret).prod() - 1

    annualized_return = (1 + total_return) ** (252 / len(daily_ret)) - 1
    annualized_vol = daily_ret.std() * np.sqrt(252)

    sharpe = annualized_return / annualized_vol if annualized_vol != 0 else np.nan

    cumulative = (1 + daily_ret).cumprod()
    running_max = cumulative.cummax()
    drawdown = cumulative / running_max - 1
    max_drawdown = drawdown.min()

    trades = (df["entry_signal"] != 0).sum()
    exits = (df["exit_signal"] != 0).sum()
    turnover = df["turnover"].sum()

    win_rate_daily = (daily_ret > 0).mean()

    active_days = (df["position"].abs() > 0).sum()
    active_ratio = active_days / len(df)

    avg_daily_return = daily_ret.mean()

    return {
        "pair": df["pair"].iloc[0],
        "strategy": df["strategy"].iloc[0],
        "total_return": total_return,
        "annualized_return": annualized_return,
        "annualized_vol": annualized_vol,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
        "win_rate_daily": win_rate_daily,
        "avg_daily_return": avg_daily_return,
        "turnover": turnover,
        "trades": trades,
        "exits": exits,
        "active_days": active_days,
        "active_ratio": active_ratio
    }

In [ ]:
all_strategy_results = pd.concat(
    [
        results_s1_ko,
        results_s2_ko,
        results_s3_ko,
        results_s4_ko,
        results_s1_xom,
        results_s2_xom,
        results_s3_xom,
        results_s4_xom,
    ],
    axis=0
)

performance_table = pd.DataFrame([
    compute_performance_metrics(results_s1_ko),
    compute_performance_metrics(results_s2_ko),
    compute_performance_metrics(results_s3_ko),
    compute_performance_metrics(results_s4_ko),
    compute_performance_metrics(results_s1_xom),
    compute_performance_metrics(results_s2_xom),
    compute_performance_metrics(results_s3_xom),
    compute_performance_metrics(results_s4_xom),
])
performance_table

In [ ]:

OUTPUT_DIR = "C:/Users/Serena/OneDrive - University of Illinois - Urbana/Desktop/Columbia/5291/Project/Data/output"

strategy_results_path = OUTPUT_DIR + "/strategy_results.csv"
performance_table_path = OUTPUT_DIR + "/performance_table.csv"

all_strategy_results.reset_index().to_csv(strategy_results_path, index=False)
performance_table.to_csv(performance_table_path, index=False)

print("Saved strategy results to:", strategy_results_path)
print("Saved performance table to:", performance_table_path)

In [ ]:
for name, df in {
    "S1 KO": results_s1_ko,
    "S2 KO": results_s2_ko,
    "S3 KO": results_s3_ko,
    "S4 KO": results_s4_ko,
    "S1 XOM": results_s1_xom,
    "S2 XOM": results_s2_xom,
    "S3 XOM": results_s3_xom,
    "S4 XOM": results_s4_xom,
}.items():
    print(name)
    print("trades:", (df["entry_signal"] != 0).sum())
    print("active ratio:", (df["position"].abs() > 0).mean())
    print("final cumulative return:", df["cumulative_return"].iloc[-1])
    print()

In [ ]:
#plot
import os
import matplotlib.pyplot as plt

PLOT_DIR = OUTPUT_DIR + "/backtest_plots"

def safe_name(name):
    return (
        name.lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace(":", "")
    )

def plot_cumulative_returns(results_list, title, save=True):
    fig, ax = plt.subplots(figsize=(10, 5))

    for df in results_list:
        label = df["strategy"].iloc[0]
        ax.plot(df.index, df["cumulative_return"], label=label)

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Cumulative Return")
    ax.legend()
    ax.grid(True, alpha=0.3)

    if save:
        filename = safe_name(title) + ".png"
        save_path = os.path.join(PLOT_DIR, filename)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()

In [ ]:
plot_cumulative_returns(
    [results_s1_ko, results_s2_ko, results_s3_ko, results_s4_ko],
    "KO-PEP Strategy Cumulative Returns"
)

plot_cumulative_returns(
    [results_s1_xom, results_s2_xom, results_s3_xom, results_s4_xom],
    "XOM-CVX Strategy Cumulative Returns"
)

# 4. Results and Research Question

# 5. Validation

## 5.1 in sample spread validation

In [ ]:
OUTPUT_DIR = DATA_DIR + "/validation_outputs"
FIG_DIR = OUTPUT_DIR + "/figures"

In [ ]:
''' 
For OLS:
Does static OLS produce a stable mean-reverting spread?
'''
def series_half_life(spread):
    s = spread.dropna()
    if len(s) < 30:
        return np.nan
    lag = s.shift(1).dropna()
    cur = s.loc[lag.index]
    x_ = lag.values
    y_ = (cur - lag).values
    X = np.column_stack([np.ones(len(x_)), x_])
    beta = np.linalg.lstsq(X, y_, rcond=None)[0][1]
    if beta >= 0:
        return np.nan
    return -np.log(2) / beta

def adf_pvalue(series):
    s = series.dropna()

    if len(s) < 30:
        return np.nan

    try:
        return adfuller(s)[1]
    except Exception:
        return np.nan
    
ols_validation = pd.DataFrame({
    "pair": ["KO-PEP", "XOM-CVX"],

    "OLS_alpha": [
        alpha_ols_ko_pep,
        alpha_ols_xom_cvx
    ],

    "OLS_beta": [
        beta_ols_ko_pep,
        beta_ols_xom_cvx
    ],

    "spread_mean": [
        spread_ols_ko_pep.mean(),
        spread_ols_xom_cvx.mean()
    ],

    "spread_std": [
        spread_ols_ko_pep.std(),
        spread_ols_xom_cvx.std()
    ],

    "spread_iqr": [
        spread_ols_ko_pep.quantile(0.75) - spread_ols_ko_pep.quantile(0.25),
        spread_ols_xom_cvx.quantile(0.75) - spread_ols_xom_cvx.quantile(0.25)
    ],

    "half_life": [
        series_half_life(spread_ols_ko_pep),
        series_half_life(spread_ols_xom_cvx)
    ],

    "ADF_pvalue": [
        adf_pvalue(spread_ols_ko_pep),
        adf_pvalue(spread_ols_xom_cvx)
    ],

    "count_abs_z_gt_2": [
        (zscore_ols_ko_pep.abs() > 2).sum(),
        (zscore_ols_xom_cvx.abs() > 2).sum()
    ],

    "pct_abs_z_gt_2": [
        (zscore_ols_ko_pep.abs() > 2).mean(),
        (zscore_ols_xom_cvx.abs() > 2).mean()
    ]
})

ols_validation.to_csv(OUTPUT_DIR + "/ols_validation_table.csv", index=False)
ols_validation

In [ ]:
#kalman validation table
kalman_validation = pd.DataFrame({
    "pair": ["KO-PEP", "XOM-CVX"],

    "Kalman_alpha_mean": [
        alpha_kf_ko_pep.mean(),
        alpha_kf_xom_cvx.mean()
    ],

    "Kalman_alpha_std": [
        alpha_kf_ko_pep.std(),
        alpha_kf_xom_cvx.std()
    ],

    "Kalman_beta_mean": [
        beta_kf_ko_pep.mean(),
        beta_kf_xom_cvx.mean()
    ],

    "Kalman_beta_std": [
        beta_kf_ko_pep.std(),
        beta_kf_xom_cvx.std()
    ],

    "Kalman_beta_min": [
        beta_kf_ko_pep.min(),
        beta_kf_xom_cvx.min()
    ],

    "Kalman_beta_max": [
        beta_kf_ko_pep.max(),
        beta_kf_xom_cvx.max()
    ],

    "spread_mean": [
        spread_kf_ko_pep.mean(),
        spread_kf_xom_cvx.mean()
    ],

    "spread_std": [
        spread_kf_ko_pep.std(),
        spread_kf_xom_cvx.std()
    ],

    "spread_iqr": [
        spread_kf_ko_pep.quantile(0.75) - spread_kf_ko_pep.quantile(0.25),
        spread_kf_xom_cvx.quantile(0.75) - spread_kf_xom_cvx.quantile(0.25)
    ],

    "half_life": [
        series_half_life(spread_kf_ko_pep),
        series_half_life(spread_kf_xom_cvx)
    ],

    "ADF_pvalue": [
        adf_pvalue(spread_kf_ko_pep),
        adf_pvalue(spread_kf_xom_cvx)
    ],

    "count_abs_z_gt_2": [
        (zscore_kf_ko_pep.abs() > 2).sum(),
        (zscore_kf_xom_cvx.abs() > 2).sum()
    ],

    "pct_abs_z_gt_2": [
        (zscore_kf_ko_pep.abs() > 2).mean(),
        (zscore_kf_xom_cvx.abs() > 2).mean()
    ]
})

kalman_validation.to_csv(OUTPUT_DIR + "/kalman_validation_table.csv", index=False)
kalman_validation

In [ ]:
#OLS vs Kalman comparison table
comparison_validation = pd.DataFrame({
    "pair": ["KO-PEP", "XOM-CVX"],

    "OLS_spread_std": [
        spread_ols_ko_pep.std(),
        spread_ols_xom_cvx.std()
    ],

    "Kalman_spread_std": [
        spread_kf_ko_pep.std(),
        spread_kf_xom_cvx.std()
    ],

    "Kalman_to_OLS_std_ratio": [
        spread_kf_ko_pep.std() / spread_ols_ko_pep.std(),
        spread_kf_xom_cvx.std() / spread_ols_xom_cvx.std()
    ],

    "OLS_half_life": [
        series_half_life(spread_ols_ko_pep),
        series_half_life(spread_ols_xom_cvx)
    ],

    "Kalman_half_life": [
        series_half_life(spread_kf_ko_pep),
        series_half_life(spread_kf_xom_cvx)
    ],

    "OLS_ADF_pvalue": [
        adf_pvalue(spread_ols_ko_pep),
        adf_pvalue(spread_ols_xom_cvx)
    ],

    "Kalman_ADF_pvalue": [
        adf_pvalue(spread_kf_ko_pep),
        adf_pvalue(spread_kf_xom_cvx)
    ]
})

comparison_validation.to_csv(OUTPUT_DIR + "/ols_vs_kalman_validation_table.csv", index=False)
comparison_validation

In [ ]:
#plots
def plot_spread(pair_name, ols_spread, kalman_spread):
    plt.figure(figsize=(12, 5))
    plt.plot(ols_spread.index, ols_spread, label="OLS spread")
    plt.plot(kalman_spread.index, kalman_spread, label="Kalman spread")
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title(f"{pair_name}: OLS vs Kalman Spread")
    plt.xlabel("Date")
    plt.ylabel("Spread")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR + f"/{pair_name}_spread_comparison.png", dpi=300)
    plt.show()

def plot_zscore(pair_name, ols_zscore, kalman_zscore):
    plt.figure(figsize=(12, 5))
    plt.plot(ols_zscore.index, ols_zscore, label="OLS z-score")
    plt.plot(kalman_zscore.index, kalman_zscore, label="Kalman z-score")
    plt.axhline(2, linestyle="--", linewidth=1)
    plt.axhline(-2, linestyle="--", linewidth=1)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title(f"{pair_name}: OLS vs Kalman Z-score")
    plt.xlabel("Date")
    plt.ylabel("Z-score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR + f"/{pair_name}_zscore_comparison.png", dpi=300)
    plt.show()

def plot_kalman_beta(pair_name, kalman_beta):
    plt.figure(figsize=(12, 5))
    plt.plot(kalman_beta.index, kalman_beta)
    plt.title(f"{pair_name}: Kalman Dynamic Hedge Ratio")
    plt.xlabel("Date")
    plt.ylabel("Beta_t")
    plt.tight_layout()
    plt.savefig(FIG_DIR + f"/{pair_name}_kalman_beta.png", dpi=300)
    plt.show()

def plot_prediction_error(pair_name, kalman_spread):
    plt.figure(figsize=(12, 5))
    plt.plot(kalman_spread.index, kalman_spread)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title(f"{pair_name}: Kalman Prediction Error / Dynamic Spread")
    plt.xlabel("Date")
    plt.ylabel("Prediction error")
    plt.tight_layout()
    plt.savefig(FIG_DIR + f"/{pair_name}_kalman_prediction_error.png", dpi=300)
    plt.show()

In [ ]:
plot_spread(
    "KO-PEP",
    spread_ols_ko_pep,
    spread_kf_ko_pep
)

plot_spread(
    "XOM-CVX",
    spread_ols_xom_cvx,
    spread_kf_xom_cvx
)

plot_zscore(
    "KO-PEP",
    zscore_ols_ko_pep,
    zscore_kf_ko_pep
)

plot_zscore(
    "XOM-CVX",
    zscore_ols_xom_cvx,
    zscore_kf_xom_cvx
)

plot_kalman_beta(
    "KO-PEP",
    beta_kf_ko_pep
)

plot_kalman_beta(
    "XOM-CVX",
    beta_kf_xom_cvx
)

plot_prediction_error(
    "KO-PEP",
    spread_kf_ko_pep
)

plot_prediction_error(
    "XOM-CVX",
    spread_kf_xom_cvx
)

## 5.2 Out-of-Sample Spread Validation

In [ ]:
OUTPUT_DIR = DATA_DIR + "/validation_outputs/oos_validation_outputs"
FIG_DIR = OUTPUT_DIR + "/figures"

In [ ]:
#Does spread remain stable in future regimes?
def adf_stat_pvalue(series):
    s = series.dropna()
    if len(s) < 30:
        return np.nan, np.nan
    result = adfuller(s)
    return result[0], result[1]

def spread_metrics(pair, model, period, spread):
    adf_stat, adf_p = adf_stat_pvalue(spread)

    return {
        "pair": pair,
        "model": model,
        "period": period,
        "spread_mean": spread.mean(),
        "spread_std": spread.std(),
        "spread_iqr": spread.quantile(0.75) - spread.quantile(0.25),
        "half_life": series_half_life(spread),
        "ADF_stat": adf_stat,
        "ADF_pvalue": adf_p,
        "n_obs": spread.dropna().shape[0]
    }

In [ ]:
# OLS OOS
def fit_static_ols(y_train, x_train):
    X = np.column_stack([np.ones(len(x_train)), x_train.values])
    Y = y_train.values
    alpha, beta = np.linalg.lstsq(X, Y, rcond=None)[0]
    return alpha, beta

def static_ols_oos_validation(
    pair,
    y,
    x,
    train_start="2010-01-01",
    train_end="2020-12-31",
    val_start="2021-01-01",
    val_end="2023-12-31",
    test_start="2024-01-01",
    test_end="2025-12-31"
):
    y = y.copy()
    x = x.copy()
    y.index = pd.to_datetime(y.index)
    x.index = pd.to_datetime(x.index)

    y_train, x_train = y.loc[train_start:train_end], x.loc[train_start:train_end]
    y_val, x_val = y.loc[val_start:val_end], x.loc[val_start:val_end]
    y_test, x_test = y.loc[test_start:test_end], x.loc[test_start:test_end]

    alpha, beta = fit_static_ols(y_train, x_train)

    spread_train = y_train - (alpha + beta * x_train)
    spread_val = y_val - (alpha + beta * x_val)
    spread_test = y_test - (alpha + beta * x_test)

    table = pd.DataFrame([
        spread_metrics(pair, "Static OLS", "train", spread_train),
        spread_metrics(pair, "Static OLS", "validation", spread_val),
        spread_metrics(pair, "Static OLS", "test", spread_test)
    ])

    table["OLS_alpha_train"] = alpha
    table["OLS_beta_train"] = beta

    return table, spread_train, spread_val, spread_test

In [ ]:
ols_ko, ols_ko_train, ols_ko_val, ols_ko_test = static_ols_oos_validation(
    "KO-PEP",
    KO_log_prices["KO"],
    PEP_log_prices["PEP"]
)

ols_xom, ols_xom_train, ols_xom_val, ols_xom_test = static_ols_oos_validation(
    "XOM-CVX",
    XOM_log_prices["XOM"],
    CVX_log_prices["CVX"]
)

In [ ]:
#kalman OOS
def kalman_oos_validation_from_existing(
    pair,
    spread_kf,
    beta_kf,
    alpha_kf,
    train_start="2010-01-01",
    train_end="2020-12-31",
    val_start="2021-01-01",
    val_end="2023-12-31",
    test_start="2024-01-01",
    test_end="2025-12-31"
):
    spread_kf.index = pd.to_datetime(spread_kf.index)
    beta_kf.index = pd.to_datetime(beta_kf.index)
    alpha_kf.index = pd.to_datetime(alpha_kf.index)

    spread_train = spread_kf.loc[train_start:train_end]
    spread_val = spread_kf.loc[val_start:val_end]
    spread_test = spread_kf.loc[test_start:test_end]

    table = pd.DataFrame([
        spread_metrics(pair, "Kalman", "train", spread_train),
        spread_metrics(pair, "Kalman", "validation", spread_val),
        spread_metrics(pair, "Kalman", "test", spread_test)
    ])

    table["Kalman_alpha_mean"] = alpha_kf.mean()
    table["Kalman_beta_mean"] = beta_kf.mean()
    table["Kalman_beta_std"] = beta_kf.std()
    table["Kalman_beta_min"] = beta_kf.min()
    table["Kalman_beta_max"] = beta_kf.max()

    return table, spread_train, spread_val, spread_test

In [ ]:
kf_ko_oos, kf_ko_train, kf_ko_val, kf_ko_test = kalman_oos_validation_from_existing(
    "KO-PEP",
    spread_kf_ko_pep,
    beta_kf_ko_pep,
    alpha_kf_ko_pep
)

kf_xom_oos, kf_xom_train, kf_xom_val, kf_xom_test = kalman_oos_validation_from_existing(
    "XOM-CVX",
    spread_kf_xom_cvx,
    beta_kf_xom_cvx,
    alpha_kf_xom_cvx
)

In [ ]:
oos_validation_table = pd.concat(
    [ols_ko, ols_xom, kf_ko_oos, kf_xom_oos],
    axis=0,
    ignore_index=True
)

oos_validation_table.to_csv(
    OUTPUT_DIR  + "/oos_spread_validation_table.csv",
    index=False
)

oos_validation_table

In [ ]:
def plot_oos_spread_comparison(
    pair,
    ols_train,
    ols_val,
    ols_test,
    kf_train,
    kf_val,
    kf_test,
    train_end="2020-12-31",
    val_end="2023-12-31"
):
    ols_spread = pd.concat([ols_train, ols_val, ols_test])
    kf_spread = pd.concat([kf_train, kf_val, kf_test])

    plt.figure(figsize=(12, 5))

    plt.plot(ols_spread.index, ols_spread, label="Static OLS spread")
    plt.plot(kf_spread.index, kf_spread, label="Kalman spread")

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.axvline(pd.to_datetime(train_end), linestyle="--", linewidth=1, color="red",label="Train/Val split")
    plt.axvline(pd.to_datetime(val_end), linestyle="--", linewidth=1, color = "green", label="Val/Test split")

    plt.title(f"{pair}: OOS Spread Validation")
    plt.xlabel("Date")
    plt.ylabel("Spread")
    plt.legend()
    plt.tight_layout()

    plt.savefig(FIG_DIR + f"/{pair}_oos_spread_validation.png", dpi=300)
    plt.show()

In [ ]:
plot_oos_spread_comparison(
    "KO-PEP",
    ols_ko_train,
    ols_ko_val,
    ols_ko_test,
    kf_ko_train,
    kf_ko_val,
    kf_ko_test,
    train_end="2020-12-31",
    val_end="2023-12-31"
)

plot_oos_spread_comparison(
    "XOM-CVX",
    ols_xom_train,
    ols_xom_val,
    ols_xom_test,
    kf_xom_train,
    kf_xom_val,
    kf_xom_test,
    train_end="2020-12-31",
    val_end="2023-12-31"
)